### Search Engine With Tools And Agents

In [1]:
!pip show langchain langchain-core langchain-community langchain-groq langsmith

Name: langchain
Version: 0.2.17
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: D:\Langchainprojects\Search Engine\venv\Lib\site-packages
Requires: aiohttp, langchain-core, langchain-text-splitters, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: langchain-community
---
Name: langchain-core
Version: 0.2.43
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: D:\Langchainprojects\Search Engine\venv\Lib\site-packages
Requires: jsonpatch, langsmith, packaging, pydantic, PyYAML, tenacity, typing-extensions
Required-by: langchain, langchain-chroma, langchain-community, langchain-groq, langchain-huggingface, langchain-ollama, langchain-openai, langchain-text-splitters
---
Name: langchain-community
Version: 0.2.19
Summary: Community contrib

In [21]:
## Custom Wikipedia Tool
## Replaced LangChain's built-in WikipediaQueryRun because the 'wikipedia'
## package returns HTTP 403 (missing User-Agent), causing JSONDecodeError.
## This implementation directly uses the official Wikipedia API with a
## proper User-Agent header.
import requests
from langchain.tools import Tool

def wiki_search(query: str):
    headers = {
        "User-Agent": "LangChainBot/1.0 (your_email@example.com)"
    }

    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": True,
        "exintro": True,
        "redirects": 1,
        "titles": query,
    }

    response = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params=params,
        headers=headers,
        timeout=10,
    )

    pages = response.json()["query"]["pages"]
    page = next(iter(pages.values()))

    return page.get("extract", "No Wikipedia article found.")

wiki = Tool(
    name="wikipedia",
    func=wiki_search,
    description="Use for general knowledge, people, anime, movies, companies, definitions, history, places."
)

In [22]:
import requests
import feedparser
from langchain.tools import Tool

def arxiv_search(query: str):
    url = (
        "https://export.arxiv.org/api/query?"
        f"search_query=all:{query.replace(' ', '+')}"
        "&start=0&max_results=3"
    )

    headers = {
        "User-Agent": "LangChainBot/1.0 (your_email@example.com)"
    }

    response = requests.get(url, headers=headers, timeout=10)

    feed = feedparser.parse(response.text)

    if not feed.entries:
        return "No papers found."

    papers = []

    for paper in feed.entries:
        papers.append(
            f"""Title: {paper.title}

Published: {paper.published}

Summary:
{paper.summary[:600]}
"""
        )

    return "\n\n".join(papers)

arxiv = Tool(
    name="arxiv",
    func=arxiv_search,
    description="Use ONLY for research papers, scientific publications, and academic literature."
)

In [23]:
tools=[wiki,arxiv]

In [24]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [25]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectordb = FAISS.from_documents(documents, embeddings)

retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A6A4A29B50>)

In [26]:
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name

'langsmith-search'

In [27]:
tools=[wiki,arxiv,retriever_tool]

In [28]:
tools

[Tool(name='wikipedia', description='Use for general knowledge, people, anime, movies, companies, definitions, history, places.', func=<function wiki_search at 0x000001A6A51360C0>),
 Tool(name='arxiv', description='Use ONLY for research papers, scientific publications, and academic literature.', func=<function arxiv_search at 0x000001A6A5136700>),
 Tool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000001A621CA6FC0>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A6A4A29B50>), document_prompt=PromptTemplate(input_variables=['page_content'], template='{page_content}'), document_separator='\n\n'), coroutine=functools.partial(<function _aget_relevant_documents at 0x000001A621CA7100>, retriever=VectorStoreRetriever(tags=['FAISS', 'Ol

In [29]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import openai
load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")
openai.api_key=os.getenv("OPENAI_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="Llama3-8b-8192")

In [35]:
## Prompt Template
from langchain import hub
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)
#prompt = hub.pull("hwchase17/openai-tools-agent")
prompt = hub.pull("hwchase17/react")

d:\Langchainprojects\Search Engine\venv\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [36]:
## Agents
from langchain.agents import create_react_agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_log_to_str(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], partial_variables={'tools': "wikipedia(query: str) - Use for general knowledge, people, anime, movies, companies, definitions, history, places.\narxiv(query: str) - Use ONLY for research papers, scientific publications, and academic literature.\nlangsmith-search(query: 'str', *, retriever: 'BaseRetriever' = VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A6A4A29B50>), document_prompt: 'BasePromptTemplate' = PromptTemplate(input_variables=['page_content'], template='{page_content}'), document_separator: 'str' = '\\n\\n', callbacks: 'Callbacks' = None) -> 'str' - Search any information about Langsmith ", 'tool_names': 'wikipedia, arxiv, langsmith-search'}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_c

In [37]:
## Agent Executer
from langchain.agents import  AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

In [40]:
agent_executor.invoke({"input":"Tell me about Aizen"})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


To answer this question, I need to consider who or what Aizen is. Aizen could be a reference to a character from an anime or manga series, a historical figure, or something else entirely. Given the broad possibilities, my first step should be to consult a general knowledge database.

Action: wikipedia
Action Input: AizenAizen may refer to:

Aizen Myō-ō (愛染明王), a Japanese Buddhist deity
Sousuke Aizen (藍染 惣右介), a main antagonist of the manga series BleachGiven the observation from Wikipedia, it seems that Aizen can refer to two different entities: a Japanese Buddhist deity named Aizen Myō-ō and a character from the manga series Bleach named Sousuke Aizen. Since the question is quite broad and does not specify which Aizen is being referred to, I should consider which of these is more commonly known or which one might be more relevant based on general interest.

Action: wikipedia
Action Input: Aizen Myō-ōRāgarāja (Sanskrit: रागराज) is a deity venerated in the Esoteric and Vajrayana Buddhis

{'input': 'Tell me about Aizen',
 'output': 'Aizen most commonly refers to Sousuke Aizen, a fictional character and the main antagonist of the first part of the Japanese manga and anime series Bleach, created by Tite Kubo. He is known for his role as the captain of the Fifth Division of Soul Reapers and his betrayal of the Soul Society in pursuit of power.'}

In [39]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: To find out what the paper 1706.03762 is about, I need to search for it in a database of research papers. The arxiv tool seems like the perfect fit for this task, as it allows me to search for research papers and scientific publications.

Action: arxiv
Action Input: 1706.03762Title: Attention Is All You Need

Published: 2017-06-12T17:57:34Z

Summary:
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the W


Title: GLU Variants Improve Transfo

{'input': "What's the paper 1706.03762 about?",
 'output': 'The paper 1706.03762, titled "Attention Is All You Need", proposes a new network architecture called the Transformer, which is based solely on attention mechanisms and does not use recurrence or convolutions, achieving state-of-the-art results in machine translation tasks.'}